# 02d | Merging and Reshaping

**After this lesson you can:** merge two tables on a key, explain why a merge can increase the row count, find records that are "in table A but not in table B," and convert between long and wide formats. About 45 minutes.

In [ ]:
import numpy as np
import pandas as pd
from practice_data import load_tables

users, orders, events, employees = load_tables()
print(len(users), len(orders))

## 1. merge

```python
users.merge(orders, on="user_id", how="left")
```

| `how=` | Rows kept | dplyr | SQL |
|---|---|---|---|
| `"inner"` (default) | Keys present on both sides | `inner_join` | `INNER JOIN` |
| `"left"` | All of the left table | `left_join` | `LEFT JOIN` |
| `"right"` | All of the right table | `right_join` | `RIGHT JOIN` |
| `"outer"` | Everything from both sides | `full_join` | `FULL OUTER JOIN` |

When the column names differ between the two sides, use `left_on=` and `right_on=`.

In [ ]:
inner = users.merge(orders, on="user_id", how="inner")
left = users.merge(orders, on="user_id", how="left")
print(len(inner), len(left))
left[left["order_id"].isna()]      # after a left join, users who never ordered have NaN in the order columns

## 2. Why the row count changed after merging

`users` has 10 rows; after the left join it has 18. The reason: **a user with several orders gets their row duplicated once per order.**

The rule is simple: if a row in the left table matches k rows in the right table, the result has k rows for it; if it matches none, a left join keeps 1 row filled with NaN, and an inner join drops it.

Interviewers often ask, "After a JOIN you have more rows than before. What could cause that?" The answer: **the join key is not unique in the right table**. If it's non-unique on both sides, you get a many-to-many join where the row counts multiply, which is usually a bug.

Good habit: check `len()` before and after a merge, or add `validate="one_to_many"` so pandas raises an error when the data doesn't match your expectation.

### Example: "aggregate first, then left join, then fill NaN with 0"

Exercise 1 below uses this exact pattern, so first walk through a complete example: how many **purchase events** (`event_type` is `"purchase"`) each user has, with 0 for users who have none.

- `.fillna(0)`: replaces missing values with 0 (R's `replace_na(x, 0)` / `coalesce(x, 0)`);
- `.rename(columns={"old_name": "new_name"})`: renames columns (02b, Section 7).

In [ ]:
cnt = (
    events[events["event_type"] == "purchase"]          # 1. filter first
    .groupby("user_id")["event_type"].count()           # 2. aggregate to one row per user
    .reset_index()                                      #    user_id becomes a regular column again
    .rename(columns={"event_type": "n_purchase"})       #    give it a clearer name
)
t = users[["user_id", "name"]].merge(cnt, on="user_id", how="left")   # 3. left join: all 10 users are kept
print(t)                                                # users with no purchases show NaN
t["n_purchase"] = t["n_purchase"].fillna(0)             # 4. fill NaN with 0
t

### ✏️ Exercise 1 (Write it yourself)

Compute **each user's total amount of completed orders**, with these requirements:

- All 10 users must appear;
- Users with no completed orders get an amount of 0;
- The result has three columns, `user_id`, `name`, `total`; store it in `out`.

Steps: first aggregate the orders to "one row per user," then left join onto `users`, and finally fill NaN with 0.

In [ ]:
# Write your code below

<details>
<summary><b>Hint</b></summary>

**Aggregate first, then merge**, so the right table has only one row per user and the merge still yields 10 rows. `fillna(0)` replaces NaN with 0.

</details>

<details>
<summary><b>Answer</b></summary>

```python
spend = (
    orders[orders["status"] == "completed"]
    .groupby("user_id")["amount"].sum()
    .reset_index()
    .rename(columns={"amount": "total"})
)
out = users[["user_id", "name"]].merge(spend, on="user_id", how="left")
out["total"] = out["total"].fillna(0)
out
```

</details>

> This order, "aggregate to one row first, then left join," matters a lot. Doing it the other way around (join first, then aggregate) can also give the right answer, but the intermediate table gets inflated, which is slower on large data and more error-prone.

## 3. Finding records that didn't match

Questions like "Which users have never placed an order?" are called anti-joins. Adding `indicator=True` to `merge` creates an extra column `_merge` that shows which side each row came from:

In [ ]:
chk = users.merge(orders[["user_id"]].drop_duplicates(), on="user_id", how="left", indicator=True)
chk[["user_id", "name", "_merge"]]

A shorter way is to use `.isin()` from 02b and negate it: `~` means "not" (R's `!`). Finally, `.tolist()` turns a column into a plain list. The code below finds the users who **never had a purchase event**:

In [ ]:
buyers = events[events["event_type"] == "purchase"]["user_id"]
no_buy = users[~users["user_id"].isin(buyers)]
print(no_buy["name"].tolist())

### ✏️ Exercise 2 (Write it yourself)

Put the names of users who **have never placed any order** into a list and store it in `no_order_names`.

In [ ]:
# Write your code below

<details>
<summary><b>Hint</b></summary>

The shortest version doesn't need merge: `users[~users["user_id"].isin(orders["user_id"])]`. `~` means "not," and `.tolist()` turns a column into a list.

</details>

<details>
<summary><b>Answer</b></summary>

```python
no_order_names = users[~users["user_id"].isin(orders["user_id"])]["name"].tolist()
print(no_order_names)
```

</details>

## 4. Stacking and deduplicating

- `pd.concat([df1, df2])`: stacks tables vertically (R's `rbind` / `bind_rows`), often followed by `.reset_index(drop=True)`;
- `df.drop_duplicates()`: keeps only one of each fully duplicated row;
- `df.drop_duplicates(subset=["user_id"], keep="first")`: judges duplicates by certain columns and keeps the first occurrence.

In `events`, user 1 has two login records on January 2. Deduplicate before counting daily active users, or the same person gets counted twice:

In [ ]:
logins = events[events["event_type"] == "login"]
print(len(logins), len(logins.drop_duplicates()))

Combining `subset=` and `keep=` with sorting lets you "take one row per group": sort in the order you want, then deduplicate on the grouping column and keep the first row. For example, the highest-paid person in each department:

In [ ]:
(
    employees.sort_values("salary", ascending=False)       # salary from high to low
    .drop_duplicates(subset=["dept"], keep="first")        # keep only the first row seen for each dept
    [["dept", "name", "salary"]]
)

### ✏️ Exercise 3 (Write it yourself)

Take each user's **earliest order** (by `order_date`), with `user_id`, `order_id`, `order_date`, and store it in `first_orders`.

Do it by "sort first, then deduplicate on user_id keeping the first row."

In [ ]:
# Write your code below

<details>
<summary><b>Answer</b></summary>

```python
first_orders = (
    orders.sort_values("order_date")
    .drop_duplicates(subset=["user_id"], keep="first")
    [["user_id", "order_id", "order_date"]]
)
first_orders
```

</details>

## 5. Long and wide tables

- `melt`: wide to long (`pivot_longer`);
- `pivot` / `pivot_table`: long to wide (`pivot_wider`).

In [ ]:
wide = pd.DataFrame({"name": ["Ann", "Bob"], "math": [90, 70], "stats": [85, 95]})
long = wide.melt(id_vars="name", var_name="subject", value_name="score")
print(long)
print(long.pivot(index="name", columns="subject", values="score").reset_index())

### ✏️ Exercise 4 (Write it yourself)

Compute the total amount of completed orders for each `country`, as two columns `country` and `total`, and store it in `out`. Ignore users with no country information.

In [ ]:
# Write your code below

<details>
<summary><b>Hint</b></summary>

First `users.merge(completed_orders, on="user_id")`, then `groupby("country")["amount"].sum()`. `groupby` drops groups whose key is missing by default.

</details>

<details>
<summary><b>Answer</b></summary>

```python
m = users.merge(orders[orders["status"] == "completed"], on="user_id", how="inner")
out = m.groupby("country")["amount"].sum().reset_index().rename(columns={"amount": "total"})
out
```

</details>

**🗣 Say it out loud:** After a JOIN, the result has more rows than the original table. What could cause that, and how would you check?

<details>
<summary><b>Reference answer</b></summary>

The join key isn't unique in the other table: a row that matches several rows gets duplicated. To check, see whether the join key is unique in each table (`df["key"].is_unique` or `df["key"].value_counts().max()`), and compare row counts before and after the merge. If the relationship should be one-to-one or one-to-many, aggregate or deduplicate the "many" side first, then join.

**Short version:** The join key isn't unique in the other table, so each matching row gets duplicated. I'd check key uniqueness on both sides, compare row counts before and after, and aggregate or deduplicate the many-side first.

</details>

## Summary

- `a.merge(b, on="key", how="left")`;
- More rows = the join key isn't unique on the other side; aggregate first, then merge;
- Finding non-matches: `~a["key"].isin(b["key"])` or `indicator=True`;
- `drop_duplicates(subset=...)`; `melt` / `pivot`.

Move on to **02e**.